# Explore low-lewel (public) data in HDF5 format

### Check paths and environments

In [ ]:
# Where are we?
! pwd
# Do you want to print the log messages?
print_log = True

In [ ]:
# Check for the versions of the core dependencies 
! conda list | grep ctlearn
! conda list | grep astropy
! conda list | grep ctapipe
! conda list | grep dl1-data-handler 
! conda list | grep keras
! conda list | grep tensorflow

### Let's import some plotting libraries

In [ ]:
from matplotlib import pyplot as plt
import matplotlib.animation as ani
from IPython.display import HTML
from traitlets.config.loader import Config

from ctapipe.core import run_tool
from ctapipe.tools.process import ProcessorTool
from ctapipe.utils.download import download_file

# For the DL1DH section
from dl1_data_handler.reader import DLImageReader, DLWaveformReader

# %matplotlib inline
plt.style.use("ggplot")

In [ ]:
# Download minimal test data of CTAO simulation
TESTDATA_DIR = "../testdata"
URL_PATH = "https://minio-cta.zeuthen.desy.de/dpps-testdata-public/data/ctapipe-test-data/v1.1.0"
gamma_simtel = "gamma_prod5.simtel.zst"
gamma_h5 = gamma_simtel.replace('.simtel.zst', '.r1.dl1.h5')
download_file(
    url=f"{URL_PATH}/{gamma_simtel}",
    path=f"{TESTDATA_DIR}/{gamma_simtel}",
    progress=True,
)
CTAO_gammafile = f"{TESTDATA_DIR}/{gamma_h5}"
argv = [
    f"--input={TESTDATA_DIR}/{gamma_simtel}",
    f"--output={CTAO_gammafile}",
    "--write-images",
    f"--DataWriter.write_r1_waveforms=True",
    "--SimTelEventSource.focal_length_choice=EQUIVALENT",
    "--overwrite"
]
assert run_tool(ProcessorTool(), argv=argv, cwd=TESTDATA_DIR) == 0


### Browsing through the HDF5 files via vitables
Use ViTables, a convenient GUI, to explore the data model.

In [ ]:
#! conda run -n vitables vitables {TESTDATA_DIR}/*.h5

## Exploring the data files with DL1DH
Besides vitables we can also explore the data with the reading functionality of the DL1DH. This is a cruical part to inspect the previously performed data reduction through ctapipe. With this notebook one can spot corrupted data and ensure that the input provided to CTLearn's CNNs is correct.

### Waveform reading
The basic functionality of reading waveforms for a telescope operating in monoscopic mode.

In [ ]:
# Configuration settings
telescope_types = {
    "LST": [3, 4],
    "MST_FlashCam": [8, 9, 14],
    "MST_NectarCam": [109, 112, 118, 119, 121, 122, 124],
    "SST": [43, 46, 53, 57, 92, 98],
}
# Valid options for LST and MSTs: "AxialMapper", "BicubicMapper", "BilinearMapper", "NearestNeighborMapper", "OversamplingMapper", "RebinMapper"
image_mapper_types = {
    "LST": "BilinearMapper",
    "MST_FlashCam": "OversamplingMapper",
    "MST_NectarCam": "AxialMapper",
    "SST": "SquareMapper",
}
for tel_type, allowed_tels in telescope_types.items():
    config = Config(
        {
            "TableQualityQuery": {
                "quality_criteria": [("> 50 phe", "hillas_intensity > 50")],
            },
            "DLWaveformReader": {
                "allowed_tels": allowed_tels, 
                "image_mapper_type": image_mapper_types[tel_type],
                "sequence_length": None, # Number of waveform samples, if 'None' all waveform smaples are taken 
                "sequence_position": "center", # Position of the sequence if 'sequence_length' is selected
                "cleaning_type": None, # Options: "image" (This would apply the DL1 image mask to the calibrated waveform)
                "focal_length_choice": "EQUIVALENT", # please ignored
            },
        },
    )
    waveform_mono_reader = DLWaveformReader(
        input_url_signal=[CTAO_gammafile],
        config=config,
    )
    batch = waveform_mono_reader.generate_mono_batch(0)
    if print_log:
        print(f"Print 'DLWaveformReader' object: {waveform_mono_reader}")
        print(f"Print the subarray information of {tel_type}:")
        print(waveform_mono_reader.subarray.info())
        print("Print the simulation information:")
        print("\n".join(waveform_mono_reader.simulation_info.pformat(
            max_lines=-1,
            max_width=-1
        )))
        print(f"Number of signal (gamma) events: {waveform_mono_reader.n_signal_events}")
        print("Generate the first batch and print its content:")
        print("\n".join(batch.pformat(
            max_lines=-1,
            max_width=-1
        )))
    
    cal_waveforms = batch["features"][0]

    fig, ax = plt.subplots(figsize=(5, 5))

    camera_screenshot = cal_waveforms[:, :, 0]

    im = ax.imshow(
        camera_screenshot,
        cmap="viridis",
        interpolation="nearest"
    )

    ax.grid(False)

    cbar = fig.colorbar(
        im,
        ax=ax,
        fraction=0.046,
        pad=0.04
    )
    cbar.set_label("p.e.", fontsize=10)

    time_text = ax.set_title(
        f"{tel_type}: Calibrated waveform (t = 0 ns) - dl1dh",
        fontsize=10,
    )

    def update(frame):
        camera_screenshot = cal_waveforms[:, :, frame]

        im.set_data(camera_screenshot)

        im.set_clim(
            camera_screenshot.min(),
            camera_screenshot.max()
        )

        time_text.set_text(
            f"{tel_type}: Calibrated waveform (t = {frame} ns) - dl1dh",
        )

        return im, time_text

    anim = ani.FuncAnimation(
        fig,
        update,
        frames=cal_waveforms.shape[-1],
        interval=500,
        blit=False
    )

    plt.close(fig)

    # IMPORTANT: display explicitly inside the loop
    display(HTML(anim.to_jshtml()))
    # Uncomment to save as GIF and show it your friends!
    anim.save(f"{tel_type}_camera_waveform_dl1dh.gif", writer=ani.PillowWriter(fps=500))

### Image reading
The basic functionality of reading images for a telescope operating in monoscopic mode. 

In [ ]:
# Configuration settings
telescope_types = {
    "LST": [3, 4],
    "MST_FlashCam": [8, 9, 14],
    "MST_NectarCam": [109, 112, 118, 119, 121, 122, 124],
    "SST": [43, 46, 53, 57, 92, 98],
}
# Valid options for LST and MST: "AxialMapper", "BicubicMapper", "BilinearMapper", "NearestNeighborMapper", "OversamplingMapper", "RebinMapper"
image_mapper_types = {
    "LST": "BilinearMapper",
    "MST_FlashCam": "OversamplingMapper",
    "MST_NectarCam": "AxialMapper",
    "SST": "SquareMapper",
}
for tel_type, allowed_tels in telescope_types.items():
    config = Config(
        {
            "TableQualityQuery": {
                "quality_criteria": [("> 50 phe", "hillas_intensity > 50")],
            },
            "DLImageReader": {
                "allowed_tels": allowed_tels, 
                "channels": ["image", "relative_peak_time"], # Options: ["image", "log_image", "cleaned_image", "log_cleaned_image", "peak_time", "relative_peak_time", "cleaned_peak_time", "cleaned_relative_peak_time"]
                "image_mapper_type": image_mapper_types[tel_type],
                "focal_length_choice": "EQUIVALENT", # please ignored
            },
        },
    )
    # Init the DLImageReader class with the configuration above
    mono_reader = DLImageReader(
        input_url_signal=[CTAO_gammafile],
        config=config,
    )
    batch = mono_reader.generate_mono_batch(0)
    if print_log:
        print(f"Print 'DLImageReader' object: {mono_reader}")
        print(f"Print the subarray information of {tel_type}:")
        print(mono_reader.subarray.info())
        print("Print the simulation information:")
        print("\n".join(mono_reader.simulation_info.pformat(
            max_lines=-1,
            max_width=-1
        )))
        print(f"Number of signal (gamma) events: {mono_reader.n_signal_events}")
        print("Generate the first batch and print its content:")
        print("\n".join(batch.pformat(
            max_lines=-1,
            max_width=-1
        )))
    # Plot the integrated image charges
    charge_image = batch["features"][0, :, :, 0]
    fig, ax = plt.subplots(figsize=(5, 5))
    im = ax.imshow(charge_image, cmap="viridis")
    ax.set_title(f"{tel_type} - charge image - dl1dh")
    ax.grid(False)
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("p.e.", fontsize=12)
    plt.show()
    # Plot the relative peak arrival times
    rel_peak_arrival_times = batch["features"][0, :, :, 1]
    fig, ax = plt.subplots(figsize=(5, 5))
    im_time = ax.imshow(rel_peak_arrival_times, cmap="viridis")
    ax.set_title(f"{tel_type} - relative peak arrival times - dl1dh")
    ax.grid(False)
    cbar = fig.colorbar(im_time, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("ns", fontsize=12)
    plt.show()


### Stereo reading
The basic functionality of reading images for a telescope system operating in stereoscopic mode.
For the upcoming cell, one would need to download at least one gamma (~0.7 GB) and proton (~1.4 GB) file
from the CTAO opendata folder (https://cloud.iaa.es/index.php/s/77d6MK4rGfanSKx).


In [ ]:
CTAO_opendata_gammafile = f"{TESTDATA_DIR}/gamma-diffuse_with_images_00.dl2.h5"
CTAO_opendata_protonfile = f"{TESTDATA_DIR}/proton_with_images_00.dl2.h5"

# Configuration settings
for tel_type in ["LST_LST_LSTCam", "MST_MST_NectarCam"]:
    min_tels = 4
    config = Config(
        {
            "TableQualityQuery": {
                "quality_criteria": [("> 500 phe", "hillas_intensity > 500")], # Very bright event
            },
            "DLImageReader": {
                "mode": "stereo",
                "allowed_tel_types": [tel_type], 
                "min_telescopes": min_tels,
                "channels": ["image", "relative_peak_time"], # Options: ["image", "log_image", "cleaned_image", "log_cleaned_image", "peak_time", "relative_peak_time", "cleaned_peak_time", "cleaned_relative_peak_time"]
                "image_mapper_type": "BilinearMapper", # Options: "AxialMapper", "BicubicMapper", "BilinearMapper", "NearestNeighborMapper", "OversamplingMapper", "RebinMapper"
            },
        },
    )
    # Init the DLImageReader class with the configuration above
    stereo_reader = DLImageReader(
        input_url_signal=[CTAO_opendata_gammafile],
        input_url_background=[CTAO_opendata_protonfile],
        config=config,
    )
    stereo_reader.subarray.peek()
    batch = stereo_reader.generate_stereo_batch(0)
    if print_log:
        print(f"Print 'DLImageReader' object: {stereo_reader}")
        print("Print the subarray information:")
        print(stereo_reader.subarray.info())
        print("Print the simulation information:")
        print("\n".join(stereo_reader.simulation_info.pformat(
            max_lines=-1,
            max_width=-1
        )))
        print(f"Number of signal (gamma) events: {stereo_reader.n_signal_events}")
        print(f"Number of background (proton) events: {stereo_reader.n_bkg_events}")
        print("Generate the first batch and print its content:")
        print("\n".join(batch.pformat(
            max_lines=-1,
            max_width=-1
        )))
    fig, axes = plt.subplots(
        2, len(batch["tel_id"]),
        figsize=(24, 14),
        constrained_layout=True
    )
    for t, tel_id in enumerate(batch["tel_id"]):
        # Integrated image charges
        charge_image = batch["features"][t, :, :, 0]
        im_charge = axes[0, t].imshow(
            charge_image,
            cmap="viridis"
        )
        axes[0, t].set_title(
            f"Tel-{tel_id} - Charge image"
        )
        axes[0, t].grid(False)
        cbar_charge = fig.colorbar(
            im_charge,
            ax=axes[0, t],
            fraction=0.046,
            pad=0.04
        )
        cbar_charge.set_label("p.e.", fontsize=12)
        # Relative peak arrival times
        rel_peak_arrival_times = batch["features"][t, :, :, 1]
        im_time = axes[1, t].imshow(
            rel_peak_arrival_times,
            cmap="viridis"
        )
        axes[1, t].set_title(
            f"Tel-{tel_id} - Peak arrival time"
        )
        axes[1, t].grid(False)
        cbar_time = fig.colorbar(
            im_time,
            ax=axes[1, t],
            fraction=0.046,
            pad=0.04
        )
        cbar_time.set_label("ns", fontsize=12)
    plt.show()